# Exploratory Data Analysis — FD001

Structured exploration of the NASA C-MAPSS turbofan engine degradation dataset (FD001 subset). Analysis logic is in `src/turbofan/eda/`; this notebook handles visualization.

## 1. Setup & Data Loading

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from turbofan.config.schema import load_config
from turbofan.data.loader import load_raw_train, load_raw_test, load_rul_labels
from turbofan.eda import quality, sensors, degradation

pio.templates.default = "plotly_white"
plt.rcParams["figure.figsize"] = (12, 6)

cfg = load_config(Path("configs/default.yaml"))

In [ ]:
train_df = load_raw_train(cfg.data)
test_df = load_raw_test(cfg.data)
test_rul = load_rul_labels(cfg.data)

print(f"Train: {train_df.shape[0]:,} rows, {train_df['engine_id'].nunique()} engines")
print(f"Test:  {test_df.shape[0]:,} rows, {test_df['engine_id'].nunique()} engines")
print(f"RUL labels: {len(test_rul)} engines")
train_df.head()

## 2. Data Quality

Check for missing values, constant sensors, and data types.

In [ ]:
missing = quality.find_missing_values(train_df)
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "None \u2014 dataset is complete.")

In [ ]:
constant = quality.find_constant_sensors(train_df)
print(f"Constant sensors ({len(constant)}): {constant}")
print("These carry no information and should be dropped before modeling.")

In [ ]:
dtype_summary = quality.summarize_dtypes(train_df)
dtype_summary

## 3. Operational Settings

Distribution and unique combinations of the three operational setting columns.

In [ ]:
op_cols = ["op_1", "op_2", "op_3"]
print("Unique values per operational setting:")
for col in op_cols:
    print(f"  {col}: {train_df[col].nunique()} unique values")

In [ ]:
op_combos = train_df.groupby(op_cols).size().reset_index(name="count")
print(f"Unique operating condition combinations: {len(op_combos)}")
op_combos

## 4. Sensor Distributions

Summary statistics and visual distributions for all 21 sensors.

In [ ]:
stats = sensors.compute_sensor_stats(train_df)
stats.round(3)

In [ ]:
sensor_cols = [c for c in train_df.columns if c.startswith("s_")]
non_constant = [c for c in sensor_cols if c not in constant]

fig, axes = plt.subplots(
    nrows=len(non_constant) // 3 + 1,
    ncols=3,
    figsize=(15, 4 * (len(non_constant) // 3 + 1)),
)
axes = axes.flatten()
for i, col in enumerate(non_constant):
    train_df[col].hist(bins=50, ax=axes[i], edgecolor="black", alpha=0.7)
    axes[i].set_title(col)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.suptitle("Sensor Distributions (non-constant)", y=1.02)
plt.show()

## 5. Correlation Analysis

Sensor-sensor and sensor-RUL correlations to identify informative features.

In [ ]:
corr = sensors.compute_correlation_matrix(train_df, non_constant)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(non_constant)))
ax.set_yticks(range(len(non_constant)))
ax.set_xticklabels(non_constant, rotation=45, ha="right")
ax.set_yticklabels(non_constant)
plt.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("Sensor-Sensor Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
train_with_rul = degradation.compute_rul_curves(train_df, max_rul=125)
rul = train_with_rul["rul"]

informative = degradation.select_informative_sensors(
    train_df, rul, threshold=0.1
)
print(f"Informative sensors (|corr with RUL| > 0.1): {informative}")

rul_corr = train_df[non_constant].corrwith(rul).sort_values()
fig = px.bar(
    x=rul_corr.values,
    y=rul_corr.index,
    orientation="h",
    labels={"x": "Pearson r with RUL", "y": "Sensor"},
    title="Sensor-RUL Correlation",
)
fig.show()

## 6. Degradation Trajectories

Visual inspection of how sensor readings evolve over an engine's lifetime.

In [ ]:
sample_engines = sorted(train_df["engine_id"].unique())[:5]
sample_df = train_df[train_df["engine_id"].isin(sample_engines)]

top_sensors = sorted(informative)[:4] if len(informative) >= 4 else informative

fig, axes = plt.subplots(len(top_sensors), 1, figsize=(14, 4 * len(top_sensors)))
if len(top_sensors) == 1:
    axes = [axes]
for ax, sensor in zip(axes, top_sensors):
    for eid in sample_engines:
        engine_data = sample_df[sample_df["engine_id"] == eid]
        ax.plot(engine_data["cycle"], engine_data[sensor], alpha=0.7, label=f"Engine {eid}")
    ax.set_ylabel(sensor)
    ax.legend(loc="upper left", fontsize=8)
axes[-1].set_xlabel("Cycle")
plt.suptitle("Raw Sensor Degradation (sample engines)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
smoothed = degradation.compute_sensor_trends(
    sample_df, top_sensors, window=10
)

fig, axes = plt.subplots(len(top_sensors), 1, figsize=(14, 4 * len(top_sensors)))
if len(top_sensors) == 1:
    axes = [axes]
for ax, sensor in zip(axes, top_sensors):
    for eid in sample_engines:
        engine_data = smoothed[smoothed["engine_id"] == eid]
        ax.plot(engine_data["cycle"], engine_data[sensor], alpha=0.7, label=f"Engine {eid}")
    ax.set_ylabel(f"{sensor} (smoothed)")
    ax.legend(loc="upper left", fontsize=8)
axes[-1].set_xlabel("Cycle")
plt.suptitle("Smoothed Sensor Trends (window=10)", y=1.01)
plt.tight_layout()
plt.show()

## 7. Summary & Key Findings

**Data quality:**
- No missing values in FD001 training data
- Several constant sensors identified (carry no information)

**Informative sensors:**
- Sensors with strong RUL correlation should be prioritized for feature engineering
- See the sensor-RUL correlation bar chart above for the full ranking

**Degradation patterns:**
- Clear monotonic degradation trends visible in informative sensors
- Rolling-mean smoothing reveals underlying signal beneath cycle-to-cycle noise

**Next steps:**
- Drop constant sensors before modeling
- Engineer rolling statistics and trend features from informative sensors
- Normalize by operational condition if multiple regimes exist in FD002+